# Dataset 10: Benchmark Indices — Data Cleaning

## Objective

Load, inspect, and clean benchmark index data while preserving the original values.

## Source File

`data/raw/10_benchmark_indices.csv`

## Output File

`data/processed/cleaned_benchmark_indices.csv`

## Workflow

1. Load and inspect the dataset.
2. Check column names and data types.
3. Identify missing values and duplicate records.
4. Validate dates and numeric index values.
5. Investigate potential data-quality issues.
6. Export and verify the cleaned dataset.


In [1]:
# Step 2: Load and inspect benchmark indices

import pandas as pd
from pathlib import Path

project_path = Path.cwd().parent

input_file = (
    project_path / "data" / "raw" / "10_benchmark_indices.csv"
)

benchmark_df = pd.read_csv(input_file)

print("Dataset loaded successfully!")
print("Dataset shape:", benchmark_df.shape)

print("\nColumn names:")
print(benchmark_df.columns.tolist())

print("\nData types:")
print(benchmark_df.dtypes)

print("\nFirst 5 rows:")
display(benchmark_df.head())

Dataset loaded successfully!
Dataset shape: (8050, 3)

Column names:
['date', 'index_name', 'close_value']

Data types:
date               str
index_name         str
close_value    float64
dtype: object

First 5 rows:


,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [2]:
# Step 3: Benchmark indices data quality checks

print("DATA QUALITY REPORT")
print("-" * 35)

print("Missing values:\n", benchmark_df.isna().sum())
print("\nDuplicate rows:", benchmark_df.duplicated().sum())

print("\nEmpty text values:")
for col in benchmark_df.select_dtypes(include="str").columns:
    print(col, ":", benchmark_df[col].str.strip().eq("").sum())

# Validate dates
benchmark_df["date"] = pd.to_datetime(
    benchmark_df["date"], errors="coerce"
)

print("\nInvalid dates:", benchmark_df["date"].isna().sum())
print("Date range:", benchmark_df["date"].min(), "to", benchmark_df["date"].max())

# Validate closing values
print("\nClosing value summary:")
display(benchmark_df["close_value"].describe())

print("Non-positive closing values:", (benchmark_df["close_value"] <= 0).sum())

print("\nIndex names and record counts:")
print(benchmark_df["index_name"].value_counts())

print("\nDuplicate date-index combinations:",
      benchmark_df.duplicated(["date", "index_name"]).sum())

DATA QUALITY REPORT
-----------------------------------
Missing values:
 date           0
index_name     0
close_value    0
dtype: int64

Duplicate rows: 0

Empty text values:
date : 0
index_name : 0

Invalid dates: 0
Date range: 2022-01-03 00:00:00 to 2026-05-29 00:00:00

Closing value summary:


count     8050.000000
mean     18351.950181
std      14169.995121
min       1444.130000
25%       2819.062500
50%      18392.960000
75%      26539.097500
max      79075.390000
Name: close_value, dtype: float64

Non-positive closing values: 0

Index names and record counts:
index_name
NIFTY50            1150
NIFTY100           1150
NIFTY_MIDCAP150    1150
BSE_SMALLCAP       1150
NIFTY500           1150
CRISIL_LIQUID      1150
CRISIL_GILT        1150
Name: count, dtype: int64

Duplicate date-index combinations: 0


In [3]:
# Step 4: Clean, export, and verify benchmark indices

# Standardize text fields
benchmark_df["index_name"] = benchmark_df["index_name"].str.strip()

# Ensure correct data types
benchmark_df["date"] = pd.to_datetime(benchmark_df["date"], errors="raise")
benchmark_df["close_value"] = pd.to_numeric(
    benchmark_df["close_value"], errors="raise"
)

# Sort records for consistent analysis
benchmark_df = benchmark_df.sort_values(
    ["index_name", "date"]
).reset_index(drop=True)

# Export cleaned dataset
output_file = project_path / "data" / "processed" / "cleaned_benchmark_indices.csv"
output_file.parent.mkdir(parents=True, exist_ok=True)

benchmark_df.to_csv(output_file, index=False)

# Reload and verify exported file
verified_df = pd.read_csv(output_file, parse_dates=["date"])

print("CLEANING & EXPORT REPORT")
print("-" * 35)
print("Output:", output_file)
print("Shape:", verified_df.shape)
print("\nMissing values:\n", verified_df.isna().sum())
print("Duplicate rows:", verified_df.duplicated().sum())
print(
    "Duplicate date-index combinations:",
    verified_df.duplicated(["date", "index_name"]).sum()
)
print("\nRecords per index:\n", verified_df["index_name"].value_counts())
print("\nExport verification:", "PASSED" if verified_df.shape == benchmark_df.shape else "FAILED")

CLEANING & EXPORT REPORT
-----------------------------------
Output: d:\Mutual-Fund-Analytics\data\processed\cleaned_benchmark_indices.csv
Shape: (8050, 3)

Missing values:
 date           0
index_name     0
close_value    0
dtype: int64
Duplicate rows: 0
Duplicate date-index combinations: 0

Records per index:
 index_name
BSE_SMALLCAP       1150
CRISIL_GILT        1150
CRISIL_LIQUID      1150
NIFTY100           1150
NIFTY50            1150
NIFTY500           1150
NIFTY_MIDCAP150    1150
Name: count, dtype: int64

Export verification: PASSED


## ✅ Data Cleaning Summary — Benchmark Indices

### Dataset Overview

* **Source:** `data/raw/10_benchmark_indices.csv`
* **Output:** `data/processed/cleaned_benchmark_indices.csv`
* **Records:** 8,050
* **Columns:** 3
* **Benchmark indices:** 7
* **Date range:** January 3, 2022 – May 29, 2026

### Cleaning Operations

* Converted `date` to datetime format.
* Converted `close_value` to numeric format.
* Removed leading and trailing whitespace from `index_name`.
* Sorted records by benchmark index and date.
* Exported the cleaned dataset without an extra index column.

### Data Quality Results

* Missing values: 0
* Duplicate rows: 0
* Duplicate date-index combinations: 0
* Invalid dates: 0
* Non-positive closing values: 0
* Export verification: PASSED

### Final Status

**Dataset 10 successfully cleaned, exported, and verified.** The cleaned dataset is ready for benchmark performance analysis and integration into the Mutual Fund Analytics project.
